1. Execute the imports


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import multiprocessing as mp
import time
from pathlib import Path

1.1 Kernel + baseline

In [4]:
# Global parameters
XMIN, XMAX = -2.0, 1.0
YMIN, YMAX = -1.5, 1.5


def mandelbrot_rows(y_start, y_end, width, height, max_iter,
                    xmin=XMIN, xmax=XMAX, ymin=YMIN, ymax=YMAX):

    xs = np.linspace(xmin, xmax, width, dtype=np.float64)
    ys = np.linspace(ymin, ymax, height, dtype=np.float64)[y_start:y_end]

    c = xs[None, :] + 1j * ys[:, None]
    z = np.zeros_like(c, dtype=np.complex128)

    counts = np.zeros(c.shape, dtype=np.int32)
    active = np.ones(c.shape, dtype=bool)

    for i in range(max_iter):
        z[active] = z[active] * z[active] + c[active]

        escaped = np.abs(z) > 2.0
        newly_escaped = escaped & active

        counts[newly_escaped] = i + 1
        active &= ~escaped

        if not active.any():
            break

    counts[active] = max_iter
    return y_start, counts


def mandelbrot_single(width, height, max_iter,
                      xmin=XMIN, xmax=XMAX, ymin=YMIN, ymax=YMAX):
    _, out = mandelbrot_rows(
        0, height, width, height, max_iter,
        xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax
    )
    return out

2. Multiprocessing version + chunking per rows.

In [5]:
def mandelbrot_multiprocessing(width, height, max_iter, processes=4, chunk_rows=64,
                               xmin=XMIN, xmax=XMAX, ymin=YMIN, ymax=YMAX,
                               force_fork=True):
    """
    Divides the image into horizontal chunks and computes each chunk in parallel using multiprocessing.
    """
    tasks = []
    for y0 in range(0, height, chunk_rows):
        y1 = min(y0 + chunk_rows, height)
        tasks.append((y0, y1, width, height, max_iter, xmin, xmax, ymin, ymax))

    if force_fork:
        try:
            ctx = mp.get_context("fork")
        except ValueError:
            ctx = mp.get_context()
    else:
        ctx = mp.get_context()

    out = np.empty((height, width), dtype=np.int32)

    with ctx.Pool(processes=processes) as pool:
        results = pool.starmap(mandelbrot_rows, tasks, chunksize=1)

    for y0, block in results:
        out[y0:y0 + block.shape[0], :] = block

    return out


def time_function(func, *args, repeats=3, **kwargs):
    """
    Returns the best time (minimum) taken by func(*args, **kwargs) over the specified number of repeats.
    """
    times = []
    result = None

    for _ in range(repeats):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        t1 = time.perf_counter()
        times.append(t1 - t0)

    return min(times), result

3. Build the benchmarks

In [6]:
def benchmark_multiprocessing(
    sizes=(1024, 2048, 4096),
    max_iter=200,
    process_list=(1, 2, 4, 8),
    chunk_rows_list=(8, 16, 32, 64, 128, 256, 512),
    repeats=3,
    save_csv=True,
    csv_path="mp_results.csv"
):
    """
    Run benchmarks for different sizes, processes and chunk_rows.
    Save times and then calculate speed-up with respect to the best case with P=1.
    """
    rows = []

    for N in sizes:
        print(f"\n===== N = {N} =====")

        for P in process_list:
            for chunk_rows in chunk_rows_list:
                if chunk_rows > N:
                    continue

                elapsed, _ = time_function(
                    mandelbrot_multiprocessing,
                    N, N, max_iter,
                    processes=P,
                    chunk_rows=chunk_rows,
                    repeats=repeats
                )

                rows.append({
                    "N": N,
                    "max_iter": max_iter,
                    "P": P,
                    "chunk_rows": chunk_rows,
                    "time_s": elapsed
                })

                print(f"N={N:5d} | P={P:2d} | chunk_rows={chunk_rows:4d} | time={elapsed:.4f}s")

    df = pd.DataFrame(rows)

    # Best chunk for each combination (N, P)
    best_idx = df.groupby(["N", "P"])["time_s"].idxmin()
    best_df = df.loc[best_idx].copy().sort_values(["N", "P"]).reset_index(drop=True)

    # Baseline = best time with P=1 for each N
    baseline = (
        best_df[best_df["P"] == 1][["N", "time_s"]]
        .rename(columns={"time_s": "baseline_time_s"})
    )

    best_df = best_df.merge(baseline, on="N", how="left")
    best_df["speedup"] = best_df["baseline_time_s"] / best_df["time_s"]

    if save_csv:
        df.to_csv(csv_path, index=False)
        best_df.to_csv(Path(csv_path).with_name("mp_best_results.csv"), index=False)

    return df, best_df

3.1 Execute benchmarks

In [ ]:
sizes = [1024, 2048, 4096]
max_iter = 200
process_list = [1, 2, 4, 8]
chunk_rows_list = [8, 16, 32, 64, 128, 256, 512]
repeats = 3

df_all, df_best = benchmark_multiprocessing(
    sizes=sizes,
    max_iter=max_iter,
    process_list=process_list,
    chunk_rows_list=chunk_rows_list,
    repeats=repeats,
    save_csv=True,
    csv_path="mp_results.csv"
)

print("\n--- All results ---")
display(df_all)

print("\n--- Best chunk for each (N, P) ---")
display(df_best)


===== N = 1024 =====
N= 1024 | P= 1 | chunk_rows=   8 | time=1.8950s
N= 1024 | P= 1 | chunk_rows=  16 | time=1.5763s
N= 1024 | P= 1 | chunk_rows=  32 | time=1.4255s
N= 1024 | P= 1 | chunk_rows=  64 | time=1.3635s
N= 1024 | P= 1 | chunk_rows= 128 | time=1.3401s
N= 1024 | P= 1 | chunk_rows= 256 | time=2.0190s
N= 1024 | P= 1 | chunk_rows= 512 | time=2.0744s
N= 1024 | P= 2 | chunk_rows=   8 | time=1.9401s
N= 1024 | P= 2 | chunk_rows=  16 | time=1.5989s
N= 1024 | P= 2 | chunk_rows=  32 | time=1.4635s
N= 1024 | P= 2 | chunk_rows=  64 | time=1.3663s
N= 1024 | P= 2 | chunk_rows= 128 | time=1.4909s
N= 1024 | P= 2 | chunk_rows= 256 | time=1.9028s
N= 1024 | P= 2 | chunk_rows= 512 | time=1.9465s
N= 1024 | P= 4 | chunk_rows=   8 | time=1.7807s
N= 1024 | P= 4 | chunk_rows=  16 | time=1.5898s
N= 1024 | P= 4 | chunk_rows=  32 | time=1.4835s
N= 1024 | P= 4 | chunk_rows=  64 | time=1.4482s
N= 1024 | P= 4 | chunk_rows= 128 | time=1.4750s
N= 1024 | P= 4 | chunk_rows= 256 | time=2.0267s
N= 1024 | P= 4 | c

4. Graphs for optimal chunk and speed-up

In [ ]:
def plot_best_chunk_vs_processes(df_best, N):
    sub = df_best[df_best["N"] == N].sort_values("P")

    plt.figure(figsize=(7, 4))
    plt.plot(sub["P"], sub["chunk_rows"], marker="o")
    plt.xlabel("Número de procesos (P)")
    plt.ylabel("Chunk size óptimo (filas)")
    plt.title(f"Chunk size óptimo vs P (N={N})")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_time_vs_processes(df_best, N):
    sub = df_best[df_best["N"] == N].sort_values("P")

    plt.figure(figsize=(7, 4))
    plt.plot(sub["P"], sub["time_s"], marker="o")
    plt.xlabel("Número de procesos (P)")
    plt.ylabel("Tiempo (s)")
    plt.title(f"Tiempo vs P usando el mejor chunk (N={N})")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_speedup_vs_processes(df_best, N):
    sub = df_best[df_best["N"] == N].sort_values("P")

    plt.figure(figsize=(7, 4))
    plt.plot(sub["P"], sub["speedup"], marker="o", label="Speed-up medido")
    plt.plot(sub["P"], sub["P"], linestyle="--", label="Ideal")
    plt.xlabel("Número de procesos (P)")
    plt.ylabel("Speed-up")
    plt.title(f"Speed-up vs P usando el mejor chunk (N={N})")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
# Fixed size example
N_plot = 4096

plot_best_chunk_vs_processes(df_best, N_plot)
plot_time_vs_processes(df_best, N_plot)
plot_speedup_vs_processes(df_best, N_plot)